# 01 — Dataset audit and immutable frame cache
Purpose: inventory the uploaded `zhanto-highlights` videos without models. Inputs: attached video Dataset, configurable root, later a manually reviewed manifest and scene cuts. No labels are fabricated. CPU decode and file hashing only. GPU accelerator is unnecessary here. Full-video hashing is linear disk I/O; dense caching is limited to explicit short segments.

Installation: use the base profile only if Kaggle lacks these utilities. Source: this laboratory's `dataset.py`; no model/checkpoint required. Review `reports/DATASET_AND_ANNOTATIONS.md` before quantitative work.

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
# Attach the unpacked laboratory as a Kaggle Dataset. Set LAB_ROOT if discovery is ambiguous.
LAB_ROOT = None
candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('README.md') if (p.parent/'scoutra_lab'/'common.py').is_file()}) if Path('/kaggle/input').exists() else []
if LAB_ROOT is None:
    if len(candidates) != 1: raise RuntimeError(f'Set LAB_ROOT to the attached laboratory folder. Candidates: {candidates}')
    LAB_ROOT = candidates[0]
LAB_ROOT = Path(LAB_ROOT)
assert (LAB_ROOT/'scoutra_lab'/'common.py').is_file()
sys.path.insert(0, str(LAB_ROOT))
WORK = Path('/kaggle/working/scoutra')
WORK.mkdir(parents=True, exist_ok=True)
print('Laboratory:', LAB_ROOT, 'Outputs:', WORK)


In [ ]:
INSTALL = False
PROFILE = 'base.txt'
if INSTALL:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(LAB_ROOT/'configs'/'profiles'/PROFILE)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'check'])
    (WORK/'requirements-resolved.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
else:
    print('Installation skipped. Direct pins are provisional until pip check passes in this Kaggle image.')


In [ ]:
from scoutra_lab.common import load_config, dump
from scoutra_lab.dataset import discover, read_manifest, validate_split, cache_segments
CFG = load_config(LAB_ROOT/'configs/dataset.yaml')
DATASET_ROOT = Path(CFG['dataset_root']) # user-supplied mount; confirm it in this Kaggle session
print('Root exists:', DATASET_ROOT.exists())
if not DATASET_ROOT.exists():
    print('Available mounts:', list(Path('/kaggle/input').iterdir()))
RUN_AUDIT = False  # set True for CPU inventory; does not run model inference
if RUN_AUDIT:
    rows = discover(DATASET_ROOT, WORK/'dataset')
    print('Video count:', len(rows))
    print(rows[:3])


## Review before caching
Open `manifest.csv`, identify the actual source match of each highlight, and set match-disjoint development/validation/test groups. Until known, leave `unassigned` and report exploratory results only. Review edits/cuts and replayed action. Replays must not create duplicate test events. Assign camera/challenge fields. Highlight clips cannot establish 90-minute continuity or long-absence recovery.

Select one **20–60 second** interval after checking duration. `clip_id` is the filename stem, not an invented sequential ID. Start with one video, not the entire dataset. Cache uses consecutive native-resolution PNG frames (no FPS subsampling); timestamps are explicitly `frame_index/fps_ESTIMATE`. VFR or precise event timing requires a later PTS-aware decoder.

In [ ]:
# Edit after inspecting actual video metadata. This empty selection intentionally blocks caching.
CFG['segments'] = []
# Example structure only — replace all values from the audit, not assumed filenames/durations:
# CFG['segments'] = [{'clip_id': 'VERIFIED_STEM', 'start_s': 0.0,
#     'duration_s': 20.0, 'scene_boundaries_reviewed': True, 'scene_cut_frames': []}]
RUN_CACHE = False
if RUN_CACHE:
    split_issues = validate_split(read_manifest(CFG['manifest']))
    print('Split warnings:', split_issues)
    frames = cache_segments(CFG)
    print('Cached frames:', len(frames))
    dump(WORK/'dataset'/'selected_cache_config.json', CFG)


## Annotation workflow
Run `python -m scoutra_lab.annotation_tool --frames .../frames.json --output .../annotation.html --every 30` in a CPU cell, then download/open the HTML. Draw exhaustive on-pitch person and ball boxes on selected frames; record physical IDs consistently. Export JSON, independently review it, and set completeness flags only after review. Dense tracking evaluation requires every frame of a short sequence, so repeat with `--every 1` and a small segment.

Upload reviewed labels as a separate private Kaggle Dataset; copy them to `/kaggle/working/scoutra/annotations/`. See report for identity anchors, crop labels and official MOT export. Annotation HTMLs embed actual frames: respect footage access when sharing.

## Outputs and troubleshooting
Each new experiment ID creates `config.yaml`, `metadata.json`, `metrics.json`, `predictions.parquet`, `runtime.json`, `errors.json`, `report.md` and `visualizations/` below `/kaggle/working/scoutra/results/`. Existing IDs are never overwritten. Unexecuted predictions are an empty **schema container**, not measured empty detections; inspect status first. Missing annotations leave metrics null.

Download the entire results directory using Kaggle **Save Version → Save & Run All** only after flags are intentional, then download notebook outputs. Also save manifest, cache configuration, annotations, hashes and resolved environment. Do not enable all notebooks or all videos together.

- Missing `/kaggle/input/...`: print the mount contents and set the verified root. Dataset titles and mount paths can differ.
- No CUDA / unsupported GPU: stop; never silently move inference to this cloud or a local machine.
- Out of memory: reduce batch/frame/pixel budget in a new recorded configuration. The second T4 is not pooled memory.
- Missing/hash-mismatched weight: attach the exact reviewed checkpoint; do not auto-fetch a similarly named model.
- Dependency/ABI error: use a fresh Kaggle session and the method-specific profile. Preserve the failed `pip check`/freeze log. Do not upgrade all packages in place.
- Missing metrics: label and review the required frames. Visual inspection is not an accuracy result.
- Source/package API mismatch: preserve the error and version; do not patch the algorithm silently.

No execution or benchmark results are embedded in this notebook.
